In [84]:
# ============================================================
# BACK-END OF COMPILER USING FLEX AND BISON
# TAC -> 8086 ASSEMBLY LANGUAGE
# Google Colab - Single Cell
# ============================================================

# Install FLEX, BISON and GCC
!apt-get update -qq
!apt-get install -y flex bison gcc -qq


# ============================================================
# Create backend.l
# ============================================================

with open("backend.l", "w") as f:
    f.write(r'''
%{
#include "backend.tab.h"
#include <string.h>
#include <stdlib.h>
%}

%option noyywrap

%%

[a-zA-Z][a-zA-Z0-9]* {
    yylval.str = strdup(yytext);
    return ID;
}

"="     { return '='; }
"+"     { return '+'; }
"-"     { return '-'; }
"*"     { return '*'; }
"/"     { return '/'; }
";"     { return ';'; }

[ \t\n]+ {
    /* Ignore whitespace */
}

. {
    return yytext[0];
}

%%
''')


# ============================================================
# Create backend.y
# ============================================================

with open("backend.y", "w") as f:
    f.write(r'''
%{
#include <stdio.h>
#include <string.h>
#include <stdlib.h>

int yylex(void);
int yyerror(char *s);
%}

%union {
    char *str;
}

%token <str> ID
%type <str> expr

%left '+' '-'
%left '*' '/'

%%

stmt_list:
      stmt_list stmt
    | stmt
    ;

stmt:
    ID '=' expr ';'
    {
        printf("MOV %s, AX\n", $1);
    }
    ;

expr:
      ID
      {
          printf("MOV AX, %s\n", $1);
          $$ = $1;
      }

    | expr '+' ID
      {
          printf("ADD AX, %s\n", $3);
          $$ = $3;
      }

    | expr '-' ID
      {
          printf("SUB AX, %s\n", $3);
          $$ = $3;
      }

    | expr '*' ID
      {
          printf("MUL %s\n", $3);
          $$ = $3;
      }

    | expr '/' ID
      {
          printf("MOV DX, 0\n");
          printf("MOV BX, %s\n", $3);
          printf("DIV BX\n");
          $$ = $3;
      }
    ;

%%

int main()
{
    printf("Enter TAC statements (end with Ctrl+D):\n");
    yyparse();

    return 0;
}

int yyerror(char *s)
{
    printf("Syntax Error: %s\n", s);
    return 0;
}
''')


# ============================================================
# Remove old generated files
# ============================================================

!rm -f backend.tab.c backend.tab.h lex.yy.c backend


# ============================================================
# Generate BISON and FLEX files
# ============================================================

!bison -d backend.y
!flex backend.l


# ============================================================
# Compile
# ============================================================

!gcc lex.yy.c backend.tab.c -o backend -lfl


# ============================================================
# SAMPLE INPUT
# ============================================================

with open("input.txt", "w") as f:
    f.write("""t1 = a + b;
t2 = t1 - c;
t3 = t2 * d;
t4 = t3 / e;
x = t4;
""")


# ============================================================
# Execute
# ============================================================

import subprocess

result = subprocess.run(
    ["./backend"],
    stdin=open("input.txt", "r"),
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Enter TAC statements (end with Ctrl+D):
MOV AX, a
ADD AX, b
MOV t1, AX
MOV AX, t1
SUB AX, c
MOV t2, AX
MOV AX, t2
MUL d
MOV t3, AX
MOV AX, t3
MOV DX, 0
MOV BX, e
DIV BX
MOV t4, AX
MOV AX, t4
MOV x, AX

